# 🎯 Ask Questions About Any PDF — Locally
**Zero API keys. Zero cost. Runs entirely on your laptop.**

## What you'll have by the end of this notebook:
- Drop any PDF → it gets chunked and embedded into ChromaDB
- Ask any question → Ollama (local LLM) answers using only your PDF

## Stack
| Component | Tool | Cost |
|---|---|---|
| LLM | Ollama (Mistral) | Free, local |
| Embeddings | sentence-transformers | Free, local |
| Vector DB | ChromaDB | Free, local |
| Framework | LangChain | Free |

## Before running this notebook, make sure:
1. Ollama is installed → [ollama.com](https://ollama.com)
2. You have pulled a model → open your terminal and run: `ollama pull mistral`
3. Ollama is running → open your terminal and run: `ollama serve`

That's it. Let's build.

---
## 📦 CELL 1 — Install all libraries

**What this does:** Installs every Python package this notebook needs.

- `langchain` + `langchain-community` — the RAG framework that connects everything
- `langchain-huggingface` — lets LangChain use HuggingFace embedding models
- `chromadb` — the local vector database where your PDF chunks get stored
- `sentence-transformers` — downloads and runs the embedding model on your machine
- `pypdf` — reads and extracts text from PDF files
- `langchain-ollama` — connects LangChain to your local Ollama LLM

⚠️ This takes 1–3 minutes on first run. You only need to do this once.

In [2]:
!pip install faiss-cpu langchain-community sentence-transformers pypdf langchain-ollama langchain-huggingface -q
print('✅ Done!')

✅ Done!



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install chromadb==0.4.24 "numpy<2.0" --force-reinstall -q

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
botocore 1.31.57 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.7.0 which is incompatible.
google-api-core 2.17.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0.dev0,>=3.19.5, but you have protobuf 6.33.6 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.1.1 which is incompatible.
proto-plus 1.23.0 requires protobuf

In [5]:
!pip install onnxruntime


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

---
## ⚙️ CELL 2 — Configuration (only cell you ever need to change)

**What this does:** Sets all the settings for your RAG pipeline in one place.

- `PDF_PATH` → path to your PDF file. Change this to your actual PDF.
- `OLLAMA_MODEL` → which Ollama model to use. `mistral` is fast and good.
  Other options: `llama3.2`, `phi3`, `gemma2`
- `CHUNK_SIZE` → how many characters per chunk. 1000 is a good default.
- `CHUNK_OVERLAP` → how many characters chunks share at their edges (prevents cutting sentences mid-thought)
- `TOP_K` → how many chunks to retrieve per question
- `CHROMA_DIR` → where ChromaDB saves its data on disk

👇 **Edit `PDF_PATH` to point to your PDF before running the next cells.**

In [6]:
import os

# ─── EDIT THIS ────────────────────────────────────────────────
PDF_PATH = 'C:/Users/Yash/Downloads/Yash_GuravResume.pdf'   # <- change to your PDF path
OLLAMA_MODEL = 'mistral'     # <- or llama3.2, phi3, gemma2
# ──────────────────────────────────────────────────────────────

CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 150
TOP_K         = 4
CHROMA_DIR    = './chroma_store'
COLLECTION    = 'pdf_rag'

# Verify the PDF exists
if not os.path.exists(PDF_PATH):
    print(f'⚠️  File not found: {PDF_PATH}')
    print('   Please update PDF_PATH above and re-run this cell.')
else:
    size_kb = os.path.getsize(PDF_PATH) / 1024
    print(f'✅ PDF found: {PDF_PATH} ({size_kb:.1f} KB)')
    print(f'   Model  : {OLLAMA_MODEL}')
    print(f'   Chunks : size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}')
    print(f'   Top-K  : {TOP_K} chunks retrieved per question')

✅ PDF found: C:/Users/Yash/Downloads/Yash_GuravResume.pdf (132.9 KB)
   Model  : mistral
   Chunks : size=1000, overlap=150
   Top-K  : 4 chunks retrieved per question


---
## 📄 CELL 3 — Load the PDF

**What this does:** Reads your PDF file and converts it into LangChain Document objects.

`PyPDFLoader` goes page by page, extracts the text from each page, and wraps it
in a `Document` object that carries both the text (`page_content`) and metadata
like the page number and source filename.

After this cell: `pages` is a list — one Document per PDF page.

In [7]:
from langchain_community.document_loaders import PyPDFLoader

print(f'Loading {PDF_PATH}...')
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f'✅ Loaded {len(pages)} pages')
print(f'\n--- Page 1 preview ---')
print(pages[0].page_content[:1000])
print(f'\nMetadata: {pages[0].metadata}')

Loading C:/Users/Yash/Downloads/Yash_GuravResume.pdf...
✅ Loaded 2 pages

--- Page 1 preview ---
Yash Gurav
♂phone+91 7028981188 /envel⌢peyashgurav002@gmail.com /linkedinlinkedin.com/yashgurav002 /githubgithub.com/yashgurav002
Summary
Full-stack AI engineer with 0→1 startup experience, building web and ML-driven systems across sports and
analytics domains. Strong in React, Django, and rapid prototyping, with hands-on experience combining
engineering and operations to launch and scale early-stage products.
Education
Fr Conceicao Rodrigues College of Engineering Mumbai, India
B.E in Artificial Intelligence and Data Science ; CGPA : 9.51/10 Sep. 2020 – May 2024
Experience
Pioneer Sports Consultancy LLP Jan 2026 – Present
Founding Engineer (Tech & Operations) Mumbai, India
• Built and launched the company’s first web interface (React), establishing an online presence and supporting player
onboarding across 8+ academy locations.
• Onboarded 20+ players and supported expansion through direct

---
## ✂️ CELL 4 — Split into chunks

**What this does:** Cuts the full PDF text into smaller overlapping pieces called chunks.

Why we need this:
- An LLM can only read a limited amount of text at once (its context window)
- Instead of sending the whole PDF, we find the relevant pieces and send only those
- Smaller chunks = more precise retrieval

`RecursiveCharacterTextSplitter` tries to split on paragraph breaks first (`\n\n`),
then sentences (`\n`), then words — keeping semantic meaning intact.

The `chunk_overlap` makes adjacent chunks share some text so context isn't lost at boundaries.

After this cell: `chunks` is a list of smaller Document objects ready to be embedded.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks = splitter.split_documents(pages)

print(f'✅ Split {len(pages)} pages → {len(chunks)} chunks')
print(f'\n--- Example chunk ---')
print(chunks[0].page_content)
print(f'\nChunk length: {len(chunks[0].page_content)} characters')
print(f'From page   : {chunks[0].metadata.get("page", "?")}')

✅ Split 2 pages → 5 chunks

--- Example chunk ---
Yash Gurav
♂phone+91 7028981188 /envel⌢peyashgurav002@gmail.com /linkedinlinkedin.com/yashgurav002 /githubgithub.com/yashgurav002
Summary
Full-stack AI engineer with 0→1 startup experience, building web and ML-driven systems across sports and
analytics domains. Strong in React, Django, and rapid prototyping, with hands-on experience combining
engineering and operations to launch and scale early-stage products.
Education
Fr Conceicao Rodrigues College of Engineering Mumbai, India
B.E in Artificial Intelligence and Data Science ; CGPA : 9.51/10 Sep. 2020 – May 2024
Experience
Pioneer Sports Consultancy LLP Jan 2026 – Present
Founding Engineer (Tech & Operations) Mumbai, India
• Built and launched the company’s first web interface (React), establishing an online presence and supporting player
onboarding across 8+ academy locations.
• Onboarded 20+ players and supported expansion through direct engagement with parents, players, and turf par

---
## 🧠 CELL 5 — Load the embedding model

**What this does:** Downloads and loads `all-MiniLM-L6-v2` — a free embedding model
that converts text into vectors (lists of numbers that capture semantic meaning).

This model:
- Downloads ~90MB on first run, then cached forever
- Runs entirely on your CPU — no GPU needed
- Produces 384-dimensional vectors
- Is fast enough to embed hundreds of chunks in seconds

We use the same model for embedding chunks (at index time) AND embedding your
questions (at query time) — this is important, both must use the same model.

In [9]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from langchain_community.embeddings import SentenceTransformerEmbeddings

print('Loading embedding model...')
embedding_model = SentenceTransformerEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

test_vec = embedding_model.embed_query('hello')
print(f'✅ Ready! Dimensions: {len(test_vec)}')

Loading embedding model...


C:\Users\Yash\AppData\Local\Temp\ipykernel_16784\3778951921.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(


✅ Ready! Dimensions: 384


In [8]:
print(f'Pages: {len(pages)}')
print(f'Chunks: {len(chunks)}')

Pages: 2
Chunks: 5


---
## 🗄️ CELL 6 — Embed chunks and store in ChromaDB

**What this does:** The core indexing step — embeds every chunk and saves it to ChromaDB on disk.

For each chunk:
1. The embedding model converts its text to a 384-number vector
2. The vector + original text + metadata are saved to ChromaDB
3. ChromaDB persists everything to `./chroma_store` on your disk

Because it saves to disk, you only need to run this cell ONCE per PDF.
Next time you open this notebook, skip straight to Cell 8 (load from disk).

⏳ This takes 10–60 seconds depending on your PDF size.

In [10]:
from langchain_community.vectorstores import FAISS

print(f'Embedding {len(chunks)} chunks into FAISS...')

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model,
)

# Save to disk
vectorstore.save_local('./faiss_store')

print(f'✅ Done! {len(chunks)} chunks stored in FAISS')
print(f'   Saved to: ./faiss_store')

Embedding 5 chunks into FAISS...
✅ Done! 5 chunks stored in FAISS
   Saved to: ./faiss_store


---
## ♻️ CELL 7 — (Skip on first run) Load existing ChromaDB from disk

**What this does:** Loads a ChromaDB you already built in a previous session.

**Only run this cell if you have already run Cell 6 before.**
It skips the embedding step and loads directly from disk — much faster.

On first run → run Cell 6, skip this.
On subsequent runs → skip Cell 6, run this instead.

In [11]:
# Only run this on subsequent sessions
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    './faiss_store',
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)
print(f'✅ Loaded FAISS store from disk!')

✅ Loaded FAISS store from disk!


---
## 🔍 CELL 8 — Set up the retriever + test it

**What this does:** Creates a retriever from the vectorstore and tests it with a sample query.

`as_retriever()` wraps ChromaDB into a LangChain retriever that:
1. Takes your question as text
2. Embeds it using the same embedding model
3. Finds the TOP_K most similar chunks using cosine similarity
4. Returns those chunks as Documents

Run this cell to confirm retrieval is working BEFORE connecting the LLM.
If the retrieved chunks look relevant to your test query — you're good.
If they look random — try adjusting CHUNK_SIZE or TOP_K in Cell 2.

In [12]:
retriever = vectorstore.as_retriever(search_kwargs={'k': TOP_K})

# Test with a sample query — change this to something relevant to your PDF
test_query = 'What is this document about?'

test_docs = retriever.invoke(test_query)

print(f'Query: "{test_query}"')
print(f'Retrieved {len(test_docs)} chunks\n')

for i, doc in enumerate(test_docs):
    print(f'--- Chunk {i+1} (page {doc.metadata.get("page", "?")}) ---')
    print(doc.page_content[:300])
    print()

Query: "What is this document about?"
Retrieved 4 chunks

--- Chunk 1 (page 1) ---
Technical Skills
Languages: Java, Python, JavaScript, SQL
Frontend: React.js, Next.js, HTML/CSS, Tailwind CSS
Backend: Django, Node.js, REST APIs
Databases: MySQL, PostgreSQL, MongoDB
Tools: Git, Postman, DVC, Claude Code
Concepts: Machine Learning, MLOps
Certifications
• Oracle Cloud Certified Cert

--- Chunk 2 (page 0) ---
Yash Gurav
♂phone+91 7028981188 /envel⌢peyashgurav002@gmail.com /linkedinlinkedin.com/yashgurav002 /githubgithub.com/yashgurav002
Summary
Full-stack AI engineer with 0→1 startup experience, building web and ML-driven systems across sports and
analytics domains. Strong in React, Django, and rapid pro

--- Chunk 3 (page 0) ---
Projects
Jeev Rakshak - Animal Detection using ML | Python , CNN, YOLO v7, Transfer Learning, React
• Developed a real-time animal detection system using YOLOv7 and React, enabling live video monitoring for safety and
surveillance use cases.
UrbanSage - Smart Cit

---
## 🤖 CELL 9 — Load Ollama (your local LLM)

**What this does:** Connects LangChain to your locally running Ollama instance.

`OllamaLLM` talks to Ollama over localhost — your model runs on your machine,
no internet needed, no API key, no cost per token.

**If this cell fails:**
- Make sure Ollama is running → open a terminal and run `ollama serve`
- Make sure you pulled the model → `ollama pull mistral`
- Check available models → `ollama list`

The sanity check at the bottom confirms Ollama is responding before you build the full chain.

In [13]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model=OLLAMA_MODEL,
    temperature=0.1,   # low = more factual, less creative
)

# Sanity check — confirm Ollama is running and responding
print(f'Testing Ollama ({OLLAMA_MODEL})...')
try:
    response = llm.invoke('Reply with exactly three words: Ollama is working')
    print(f'✅ Ollama response: {response.strip()}')
except Exception as e:
    print(f'❌ Ollama error: {e}')
    print('   → Make sure Ollama is running: open a terminal and run `ollama serve`')
    print(f'   → Make sure you have the model: `ollama pull {OLLAMA_MODEL}`')

Testing Ollama (mistral)...
✅ Ollama response: Working diligently.


---
## 📝 CELL 10 — Build the prompt template

**What this does:** Defines the prompt that gets sent to the LLM for every question.

The prompt has two slots:
- `{context}` — the retrieved chunks from your PDF, joined together
- `{question}` — your actual question

The instructions tell the LLM:
- Answer ONLY from the provided context (no hallucination)
- If the answer isn't in the context, say so honestly
- Be concise

⚠️ Note: We write this prompt inline instead of using `hub.pull('rlm/rag-prompt')`
because the hub pull requires a tracing parameter that causes errors locally.
This is the exact same prompt content — just written directly.

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('human', """You are an assistant for question-answering tasks.
Use ONLY the following context to answer the question.
If the answer is not in the context, say \"I don't know based on the provided document.\"
Keep your answer concise — 3 sentences maximum.

Context:
{context}

Question: {question}

Answer:""")
])

print('✅ Prompt template ready!')
print('\nSlots: {context} and {question}')

✅ Prompt template ready!

Slots: {context} and {question}


---
## ⛓️ CELL 11 — Assemble the full RAG chain

**What this does:** Connects all the pieces into one chain using LangChain Expression Language (LCEL).

When you call `rag_chain.invoke({'question': '...'})` this is what happens in order:

```
Your question
    → retriever        finds TOP_K relevant chunks from ChromaDB
    → format_docs      joins chunks into one context string
    → prompt           fills {context} and {question} into the template
    → llm              Ollama reads the context and generates an answer
    → StrOutputParser  extracts the plain text from the response
```

The `|` pipe operator chains steps — same idea as Unix pipes.
`RunnablePassthrough()` passes your question through unchanged to the prompt.

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    """Join retrieved chunks into one context block for the LLM."""
    return '\n\n'.join(
        f'[Page {doc.metadata.get("page", "?")}]\n{doc.page_content}'
        for doc in docs
    )

rag_chain = (
    {
        'context': retriever | format_docs,
        'question': RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print('✅ RAG chain assembled!')
print('Flow: question → retriever → prompt → Ollama → answer')

✅ RAG chain assembled!
Flow: question → retriever → prompt → Ollama → answer


---
## 🚀 CELL 12 — Ask your first question!

**What this does:** Runs the full RAG pipeline end to end.

Change `my_question` to anything you want to ask about your PDF.
The answer will be grounded only in your document — the LLM cannot make things up
because the prompt explicitly restricts it to the provided context.

⏳ First question takes 5–15 seconds (Ollama loads the model into memory).
Subsequent questions are faster.

In [16]:
# ✏️ Change this to any question about your PDF
my_question = 'What is this document about?'

print(f'❓ Question: {my_question}')
print('⏳ Thinking...\n')

answer = rag_chain.invoke(my_question)

print('💬 Answer:')
print(answer)

❓ Question: What is this document about?
⏳ Thinking...

💬 Answer:
 The document is about a full-stack AI engineer named Yash Gurav who has experience in building web and machine learning (ML)-driven systems. He has worked on various projects such as Jeev Rakshak for animal detection, UrbanSage for smart city issue reporting, and Review Xpert for trend analysis using big data analytics. His technical skills include languages like Java, Python, JavaScript, SQL, and tools like Git, Postman, DVC, Claude Code. He has also worked on frontend technologies like React.js, Next.js, HTML/CSS, Tailwind CSS, and backend technologies like Django, Node.js, REST APIs. His certifications include Oracle Cloud Certified Generative AI Professional, Oracle Cloud Certified AI Foundations Associate, and Oracle Cloud Certified Foundations Associate.


---
## 🔎 CELL 13 — Ask a question AND see the source chunks

**What this does:** Same as Cell 12 but also shows you exactly which chunks from your PDF
the LLM used to build the answer.

This is useful for:
- Verifying the answer is actually grounded in your document
- Debugging if the answer looks wrong (maybe wrong chunks are being retrieved)
- Understanding which pages in your PDF contain the relevant information

If the retrieved chunks look off, go back to Cell 8 and change `test_query` to diagnose.

In [17]:
# ✏️ Change this to any question
my_question = 'What are the key points in this document?'

# Step 1: retrieve chunks
retrieved = retriever.invoke(my_question)

# Step 2: generate answer
print(f'❓ Question: {my_question}')
print('⏳ Thinking...\n')
answer = rag_chain.invoke(my_question)

print('💬 Answer:')
print(answer)

print('\n' + '='*60)
print('📄 Sources used to generate this answer:')
print('='*60)
for i, doc in enumerate(retrieved):
    page = doc.metadata.get('page', '?')
    print(f'\n[Source {i+1} — Page {page}]')
    print(doc.page_content[:400])
    if len(doc.page_content) > 400:
        print('...')

❓ Question: What are the key points in this document?
⏳ Thinking...

💬 Answer:
 The document details the technical skills of an individual named Yash Gurav, including proficiency in languages such as Java, Python, JavaScript, SQL, and specific frontend and backend technologies. He also has experience with various databases and tools. His certifications include Oracle Cloud Certified Generative AI Professional, Oracle Cloud Certified AI Foundations Associate, and Oracle Cloud Certified Foundations Associate. In terms of experience, he has worked as a Founding Engineer at Pioneer Sports Consultancy LLP and a Full Stack Developer Intern at K3Y Technology Services Pvt Ltd - Sahayata24x7. His projects include Jeev Rakshak - Animal Detection using ML, UrbanSage - Smart City Issue Reporting Platform, and Review Xpert – Trend Analysis using Big Data Analytics.

📄 Sources used to generate this answer:

[Source 1 — Page 1]
Technical Skills
Languages: Java, Python, JavaScript, SQL
Frontend: React

---
## 💬 CELL 14 — Mini chat loop (ask multiple questions)

**What this does:** Lets you ask multiple questions one after another in a simple loop
without re-running the cell each time.

Type your question and press Enter.
Type `quit` or `exit` to stop the loop.

Note: this is a simple loop — it does NOT have memory of previous questions.
Each question is answered independently from the PDF context.
(Multi-turn memory is a future upgrade from the PRD.)

In [21]:
print('📖 PDF Q&A — type your questions below.')
print('   Type "quit" or "exit" to stop.\n')

while True:
    question = input('❓ Your question: ').strip()

    if not question:
        continue

    if question.lower() in ('quit', 'exit', 'q'):
        print('👋 Bye!')
        break

    print('⏳ Thinking...\n')
    answer = rag_chain.invoke(question)
    print(f'💬 {answer}\n')
    print('-' * 60)

📖 PDF Q&A — type your questions below.
   Type "quit" or "exit" to stop.



❓ Your question:  Give a summary on food provision


⏳ Thinking...

💬  About 17% of global human consumption of animal protein comes from marine and freshwater wild-caught and aquacultured aquatic animals (FAO, 2020a; Costello et al., 2020). Some regions like Small Island Developing States have a higher per capita intake, and consumption is 15 times higher in Indigenous Peoples compared to non-Indigenous Peoples. Fishery products also supply critical dietary micronutrients worldwide (Hicks et al., 2019; Vianna et al., 2020).

------------------------------------------------------------


❓ Your question:  give summary on Observed and projected changes in contemporary  community structure and biodiversity


⏳ Thinking...

💬  The paleoecological record shows large-scale biome shifts and changes in community composition due to past climate changes, with species extinctions in some groups under RCP6 and 8.5 scenarios (high confidence). Spatial shifts of marine species due to projected warming will cause high-latitude invasions and high local-extinction rates in the tropics and semi-enclosed seas (medium confidence). Loss of corals, changes in community composition, and inter-specific interactions of zooplankton are also observed with an elevated risk on community structure in the 21st century (medium confidence).

------------------------------------------------------------


...

⏳ Thinking...

💬  Jessica Bolin

------------------------------------------------------------


❓ Your question:  who are this guys Laurent Bopp (France), Philip Boyd (Australia/UK), Simon Donner (Canada), Shin Ichi Ito (Japan), Wolfgang Kiessling (Germany), Paulina Martinetto (Argentina), Elena Ojea (Spain),  Marie-Fanny Racault (UK/France), Björn Rost (Germany), Mette Skern-Mauritzen (Norway), Dawit  Yemane Ghebrehiwet (South Africa/Eritrea


⏳ Thinking...

💬  These individuals are the Lead Authors for the document provided.

------------------------------------------------------------


❓ Your question:  quit


👋 Bye!


---
## 🔄 CELL 15 — Switch to a different PDF

**What this does:** Clears the current ChromaDB collection and indexes a new PDF from scratch.

Run this when you want to ask questions about a completely different document.
It:
1. Deletes the current ChromaDB collection
2. Loads and chunks your new PDF
3. Re-embeds and stores in ChromaDB
4. Reconnects the retriever and chain

After running this cell, Cells 12–14 will answer questions about the NEW PDF.

In [19]:
# ✏️ Set the path to your new PDF
NEW_PDF_PATH = 'C:/Users/Yash/Downloads/IPCC_AR6_WGII_Chapter03.pdf'

import shutil
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

if not os.path.exists(NEW_PDF_PATH):
    print(f'❌ File not found: {NEW_PDF_PATH}')
else:
    # Step 1: delete old FAISS store
    if os.path.exists('./faiss_store'):
        shutil.rmtree('./faiss_store')
        print(f'🗑️  Cleared old FAISS store')

    # Step 2: load + chunk new PDF
    print(f'Loading {NEW_PDF_PATH}...')
    new_pages = PyPDFLoader(NEW_PDF_PATH).load()
    new_chunks = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    ).split_documents(new_pages)
    print(f'Split into {len(new_chunks)} chunks')

    # Step 3: embed + store
    print('Embedding and storing in FAISS...')
    vectorstore = FAISS.from_documents(
        documents=new_chunks,
        embedding=embedding_model,
    )
    vectorstore.save_local('./faiss_store')

    # Step 4: reconnect retriever + chain
    retriever = vectorstore.as_retriever(search_kwargs={'k': TOP_K})
    rag_chain = (
        {'context': retriever | format_docs, 'question': RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )

    PDF_PATH = NEW_PDF_PATH
    print(f'\n✅ Ready! Now answering questions about: {NEW_PDF_PATH}')

🗑️  Cleared old FAISS store
Loading C:/Users/Yash/Downloads/IPCC_AR6_WGII_Chapter03.pdf...
Split into 1325 chunks
Embedding and storing in FAISS...

✅ Ready! Now answering questions about: C:/Users/Yash/Downloads/IPCC_AR6_WGII_Chapter03.pdf


---
## 🩺 CELL 16 — Troubleshooting reference

**Not a runnable cell — just a reference if something goes wrong.**

| Problem | Fix |
|---|---|
| `Connection refused` on Ollama | Run `ollama serve` in a terminal |
| `model not found` error | Run `ollama pull mistral` in a terminal |
| PDF loads but 0 chunks | Check `PDF_PATH` in Cell 2 is correct |
| Answers are wrong/irrelevant | Reduce `CHUNK_SIZE` to 500 or increase `TOP_K` to 6 |
| ChromaDB error on Cell 6 | Delete the `./chroma_store` folder and re-run Cell 6 |
| Embedding model download stuck | Check your internet connection — it downloads once then caches |
| Very slow answers | Switch to `phi3` model in Cell 2 — it's much faster |

**Check which Ollama models you have installed:**
```bash
ollama list
```

**Pull a different model:**
```bash
ollama pull llama3.2
ollama pull phi3
ollama pull gemma2
```